# LFM3-1.7B — chat and tools on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/lfm3-colab-notebooks-103d/colab/LFM3-1.7B-chat-tools.ipynb)

[`LiquidAI/LFM3-1.7B`](https://huggingface.co/LiquidAI/LFM3-1.7B) is Liquid AI's new **text** Liquid Foundation Model (~1.73B, `lfm3` architecture). It is the shared backbone of the [LFM3 family](https://huggingface.co/collections/LiquidAI/lfm3): a tri-hybrid of gated short convolutions, sliding-window attention, and global attention, with a **256K** context window. The card post-trains it for local agents, tool calling, and structured outputs. It is **not** the vision, audio, or omni checkpoint.

Native Transformers LFM3 support is still landing in [`huggingface/transformers#50036`](https://github.com/huggingface/transformers/pull/50036). This notebook follows the card: **Transformers + `trust_remote_code`**, not a from-source transformers pin.

**Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/LFM3-1.7B/blob/main/LICENSE) (`license: other`, `lfm1.0`). This is **not** Apache 2.0 or MIT. Read the license before any production or paid use.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has about 15 GB of VRAM, **no native bfloat16**, and **no FlashAttention-2**. The card's examples use bfloat16; this notebook loads **float16** (~3.5 GB of weights) with **SDPA** (eager if SDPA refuses the hybrid stack). Keep generations short — 256K context does not fit a T4 KV cache.

**Links:** [model card](https://huggingface.co/LiquidAI/LFM3-1.7B) · [blog](https://www.liquid.ai/blog/lfm3) · [announcement](https://x.com/liquidai/status/2115128329003303107) · [collection](https://huggingface.co/collections/LiquidAI/lfm3) · [GGUF](https://huggingface.co/LiquidAI/LFM3-1.7B-GGUF)


In [ ]:
%pip install -q -U "transformers>=5.20.2" accelerate

# The 1.7B card's Transformers quick start uses the text-generation pipeline and
# trust_remote_code (custom Lfm3ForCausalLM). There is no extra pin beyond a current
# transformers. 5.20.2 matches the LFM3-Audio / Omni cards. Do not reinstall torch —
# keep Colab's CUDA wheel.
# Native LFM3 classes may still be landing (huggingface/transformers#50036).


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


## 1. Check the T4 and pick a dtype

T4 is `sm_75`: float16, SDPA / eager, no FlashAttention-2. The next cell stops if you are on CPU. Weights are public — cancel any `HF_TOKEN` prompt.


In [ ]:
import os
import torch
import transformers
from packaging.version import Version

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

assert Version(transformers.__version__) >= Version("5.20.2"), (
    f"transformers {transformers.__version__} is loaded; this notebook needs 5.20.2+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# The cards default to bfloat16 on GPU. A T4 has no native BF16, so use float16.
major, minor = torch.cuda.get_device_capability()
DTYPE = torch.float16
DEVICE = "cuda"
props = torch.cuda.get_device_properties(0)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {props.total_memory / 1024**3:.1f} GB")
print("FlashAttention-2: off (sdpa / eager only)")
if major < 8:
    print("No native bfloat16 on this GPU; loading float16.")
if major >= 8:
    print("This GPU can run bfloat16; this notebook still uses float16 so a T4 recipe stays consistent.")


## 2. Load LFM3-1.7B

`AutoTokenizer` + `AutoModelForCausalLM` with `trust_remote_code=True`, as required by the checkpoint's `auto_map`. First run downloads about 3.5 GB of safetensors. Attention is **sdpa**, then **eager**.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "LiquidAI/LFM3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


def load_causal(attn):
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        attn_implementation=attn,
        device_map="auto",
        trust_remote_code=True,
    ).eval()


ATTN = None
model = None
last_error = None
for attn in ("sdpa", "eager"):
    try:
        model = load_causal(attn)
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} float16 load failed:", exc)
        del model
        model = None
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("could not load LFM3-1.7B") from last_error

DEVICE = next(model.parameters()).device
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print("attention:", ATTN, "| dtype:", getattr(model, "dtype", DTYPE))
print(f"parameters: {n_params:.2f}B")
print(
    f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)


## 3. Generation helper

Card sampling for chat / tools / agents: `temperature=0.1`, `top_k=50`, `repetition_penalty=1.1`. Reasoning is **on by default**; pass `enable_thinking=False` to skip the `<think>` trace (the card says this can be inconsistent on hard reasoning). T4 demos stay at **256** new tokens.


In [ ]:
import json
import re

SAMPLER = dict(do_sample=True, temperature=0.1, top_k=50, repetition_penalty=1.1)


def generate(messages, tools=None, enable_thinking=False, max_new_tokens=256):
    kwargs = dict(
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=enable_thinking,
    )
    if tools is not None:
        kwargs["tools"] = tools
    inputs = tokenizer.apply_chat_template(messages, **kwargs).to(DEVICE)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, **SAMPLER)
    raw = tokenizer.decode(
        output[0, inputs["input_ids"].shape[1] :],
        skip_special_tokens=False,
    )
    print(
        f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )
    return raw


def parse_model_output(text):
    think_match = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL)
    if think_match:
        thinking = think_match.group(1).strip()
        answer_start = text.find("</think>") + len("</think>")
        answer_end = text.find("<|im_end|>", answer_start)
        answer = (
            text[answer_start:answer_end].strip()
            if answer_end != -1
            else text[answer_start:].strip()
        )
    else:
        thinking, answer = "", text.replace("<|im_end|>", "").strip()
    return thinking, answer


def parse_tool_calls(text):
    calls = []
    for match in re.finditer(
        r"<tool_call>\s*<function=([^>]+)>(.*?)</function>\s*</tool_call>",
        text,
        flags=re.DOTALL,
    ):
        name = match.group(1).strip()
        args = {}
        for pname, pvalue in re.findall(
            r"<parameter=([^>]+)>\s*(.*?)\s*</parameter>",
            match.group(2),
            flags=re.DOTALL,
        ):
            raw_value = pvalue.strip()
            try:
                args[pname] = json.loads(raw_value)
            except json.JSONDecodeError:
                args[pname] = raw_value
        calls.append({"type": "function", "function": {"name": name, "arguments": args}})
    return calls


## 4. Demo 1 — short chat

The first prompt is the card's *What is C. elegans?* with thinking off so the T4 stays responsive. The second call leaves thinking on for a tiny factual question.


In [ ]:
print("=== chat (enable_thinking=False) ===")
raw = generate(
    [{"role": "user", "content": "What is C. elegans?"}],
    enable_thinking=False,
    max_new_tokens=256,
)
print(raw)
_, answer = parse_model_output(raw)
print()
print("--- answer ---")
print(answer)

print()
print("=== thinking (enable_thinking=True) ===")
raw_think = generate(
    [{"role": "user", "content": "How many r letters are in the word strawberry?"}],
    enable_thinking=True,
    max_new_tokens=256,
)
print(raw_think)
think, think_answer = parse_model_output(raw_think)
print()
print("--- think ---")
print(think or "(empty)")
print()
print("--- answer ---")
print(think_answer)


## 5. Demo 2 — tool-calling loop

Pass OpenAI-style function schemas through `tools=...` to `apply_chat_template()`. LFM3 writes XML calls (`<tool_call><function=name>…`). The loop is the card's recipe: append the assistant `tool_calls` plus a `role="tool"` result, keep the schemas, generate again. One mock tool: weather in Paris.


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                },
                "required": ["city"],
            },
        },
    }
]


def get_weather(city):
    table = {"paris": "16 C, light rain", "tokyo": "22 C, clear"}
    return {"city": city, "forecast": table.get(city.lower(), "18 C, cloudy")}


FUNCTIONS = {"get_weather": get_weather}

messages = [
    {"role": "user", "content": "What is the weather in Paris right now?"},
]
print("=== first turn ===")
raw = generate(messages, tools=TOOLS, enable_thinking=False, max_new_tokens=256)
print(raw)
calls = parse_tool_calls(raw)
print("parsed calls:", calls)

if not calls:
    print("Model answered without a tool call; skipping the tool-result turn.")
else:
    results = []
    for call in calls:
        fn = call["function"]["name"]
        args = call["function"]["arguments"]
        results.append(FUNCTIONS[fn](**args))
    messages.append({"role": "assistant", "content": raw, "tool_calls": calls})
    messages.append({"role": "tool", "content": json.dumps(results)})
    print()
    print("=== after tool result ===")
    raw2 = generate(messages, tools=TOOLS, enable_thinking=False, max_new_tokens=256)
    print(raw2)
    _, final = parse_model_output(raw2)
    print()
    print("--- answer ---")
    print(final)


## 6. Optional: GGUF / llama.cpp

A [`Q4_K_M`](https://huggingface.co/LiquidAI/LFM3-1.7B-GGUF/blob/main/LFM3-1.7B-Q4_K_M.gguf) (~1.1 GB) lives at [`LiquidAI/LFM3-1.7B-GGUF`](https://huggingface.co/LiquidAI/LFM3-1.7B-GGUF). The card says this needs an **LFM3-compatible llama.cpp** build:

```bash
llama-cli -hf LiquidAI/LFM3-1.7B-GGUF:Q4_K_M \
  --temp 0.1 --top-k 50 --repeat-penalty 1.1
```

This notebook does **not** install `llama-cpp-python` (that wheel often fights Colab's torch). Transformers + `trust_remote_code` is the primary T4 path. Skip GGUF unless you already have an LFM3 llama.cpp binary.


## Caveats

- **Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/LFM3-1.7B/blob/main/LICENSE) (`license: other`). Not Apache / MIT.
- **T4 caps:** float16 + SDPA, short `max_new_tokens`. The 256K window is a card feature, not a T4 demo.
- **Architecture:** custom `lfm3` code via `trust_remote_code`. Native Transformers support is still [PR #50036](https://github.com/huggingface/transformers/pull/50036).
- **HF token:** the repo is public. Cancel Colab `HF_TOKEN` prompts; this notebook sets `HF_HUB_DISABLE_IMPLICIT_TOKEN=1`.
- **Not executed here:** the agent VM has no GPU. Install cells parse; a full T4 `generate()` was not run in this environment.
- **Do not merge from this agent.** Leave the PR open for human review.
